# MB51 — Exploración interactiva PP04
**Clases:** 261 (orden de producción) · 201 (centro de coste)  
**Período:** 2019 → 2026  |  1,868,950 registros  |  Moneda: COP

> Ejecuta **Celda 0** primero (carga ~10 s). El resto responde en tiempo real.

## 0 · Carga y limpieza (ejecutar una vez)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as w
from IPython.display import display, clear_output
from pathlib import Path

PARQUET = Path('../data/mb51.parquet')
assert PARQUET.exists(), 'Ejecuta primero scripts/descarga.py'

def parse_sap_num(serie):
    s = serie.str.strip()
    negativo = s.str.endswith('-')
    valor = pd.to_numeric(s.str.rstrip('-').str.replace(',', '', regex=False), errors='coerce')
    return valor.where(~negativo, -valor)

print('Cargando...')
df = pd.read_parquet(PARQUET)
df['CANTIDAD_N'] = parse_sap_num(df['CANTIDAD'])
df['IMPORTE_N']  = parse_sap_num(df['IMPORTE'])
df['FECHA']      = pd.to_datetime(df['REGISTRADO_EL'], format='%Y%m%d', errors='coerce')
df['AÑO']        = df['FECHA'].dt.year.astype('Int64')
df['MES']        = df['FECHA'].dt.to_period('M')

AÑOS      = sorted(df['AÑO'].dropna().unique().tolist())
MATERIALES = sorted(df['TEXTO_MATE'].dropna().unique().tolist())
print(f'OK — {len(df):,} filas cargadas. Años disponibles: {AÑOS[0]}–{AÑOS[-1]}')

## 1 · Explorador de movimientos
Agrupa y filtra los movimientos. Usa los controles para cambiar la vista.

In [ ]:
# ── Controles ──────────────────────────────────────────────────────────
w_clase = w.SelectMultiple(
    options=['261', '201'], value=['261', '201'],
    description='Clase mov:', rows=2
)
w_años = w.IntRangeSlider(
    value=[AÑOS[0], AÑOS[-1]], min=AÑOS[0], max=AÑOS[-1], step=1,
    description='Años:', continuous_update=False, layout=w.Layout(width='500px')
)
w_texto = w.Text(
    value='', placeholder='ej: y3, pvb, modmed ...',
    description='Material:'
)
w_agrupar = w.Dropdown(
    options=['TEXTO_MATE', 'MATERIAL', 'AÑO', 'MES', 'CLASE_MOV', 'LOTE', 'ORDEN'],
    value='TEXTO_MATE', description='Agrupar por:'
)
w_metrica = w.Dropdown(
    options=[('M2 consumidos', 'm2'), ('Importe (GCOP)', 'gcop'), ('Movimientos', 'mov')],
    value='m2', description='Metrica:'
)
w_top = w.IntSlider(value=20, min=5, max=50, step=5, description='Top N:')
w_grafico = w.Checkbox(value=True, description='Mostrar grafico')

out = w.Output()

def actualizar(_=None):
    with out:
        clear_output(wait=True)
        # Filtros
        mask = (
            df['CLASE_MOV'].isin(list(w_clase.value)) &
            df['AÑO'].between(w_años.value[0], w_años.value[1])
        )
        if w_texto.value.strip():
            mask &= df['TEXTO_MATE'].str.contains(w_texto.value.strip(), case=False, na=False)
        sub = df[mask]
        if sub.empty:
            print('Sin resultados con los filtros actuales.')
            return
        # Agrupación
        grp_col = w_agrupar.value
        grp = sub.groupby(grp_col, observed=True).agg(
            movimientos  =('ID',         'count'),
            m2           =('CANTIDAD_N', lambda x: x.abs().sum()),
            gcop         =('IMPORTE_N',  lambda x: x.abs().sum() / 1e9),
            ordenes      =('ORDEN',      'nunique'),
            lotes        =('LOTE',       'nunique'),
        ).sort_values(w_metrica.value, ascending=False).head(w_top.value)
        # Tabla
        print(f'Registros filtrados: {len(sub):,}  |  Grupos: {len(grp)}')
        display(grp.round(4))
        # Grafico
        if w_grafico.value and len(grp) > 0:
            etiquetas = grp.index.astype(str)
            valores   = grp[w_metrica.value]
            fig, ax = plt.subplots(figsize=(12, max(4, len(grp) * 0.35)))
            ax.barh(etiquetas, valores)
            ax.invert_yaxis()
            unidad_label = {'m2': 'M2', 'gcop': 'GCOP', 'mov': 'movimientos'}[w_metrica.value]
            ax.set_xlabel(unidad_label)
            ax.set_title(f'Top {w_top.value} por {grp_col} — {unidad_label}')
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.1f}'))
            plt.tight_layout()
            plt.show()

for widget in [w_clase, w_años, w_texto, w_agrupar, w_metrica, w_top, w_grafico]:
    widget.observe(actualizar, names='value')

panel = w.VBox([
    w.HBox([w_clase, w_años]),
    w.HBox([w_texto, w_agrupar, w_metrica]),
    w.HBox([w_top, w_grafico]),
    out
])
actualizar()
panel

## 2 · Lotes por material Y3
Enfocado en los 4 materiales Y3. Selecciona el material y el rango de años.

In [ ]:
df_y3 = df[df['TEXTO_MATE'].str.contains('y3', case=False, na=False)].copy()
MATS_Y3 = sorted(df_y3['TEXTO_MATE'].unique().tolist())

w_mat_y3 = w.SelectMultiple(
    options=MATS_Y3, value=MATS_Y3,
    description='Material Y3:', rows=4, layout=w.Layout(width='300px')
)
w_años_y3 = w.IntRangeSlider(
    value=[AÑOS[0], AÑOS[-1]], min=AÑOS[0], max=AÑOS[-1], step=1,
    description='Años:', continuous_update=False, layout=w.Layout(width='500px')
)
w_agrupar_y3 = w.Dropdown(
    options=['LOTE', 'ORDEN', 'AÑO', 'MES', 'TEXTO_MATE'],
    value='LOTE', description='Agrupar por:'
)
w_metrica_y3 = w.Dropdown(
    options=[('M2 consumidos', 'm2'), ('Importe (GCOP)', 'gcop'), ('Movimientos', 'mov')],
    value='m2', description='Metrica:'
)
w_top_y3 = w.IntSlider(value=20, min=5, max=100, step=5, description='Top N:')

out_y3 = w.Output()

def actualizar_y3(_=None):
    with out_y3:
        clear_output(wait=True)
        mask = (
            df_y3['TEXTO_MATE'].isin(list(w_mat_y3.value)) &
            df_y3['AÑO'].between(w_años_y3.value[0], w_años_y3.value[1])
        )
        sub = df_y3[mask]
        if sub.empty:
            print('Sin resultados.')
            return
        grp_col = w_agrupar_y3.value
        grp = sub.groupby(grp_col, observed=True).agg(
            movimientos=('ID',         'count'),
            m2         =('CANTIDAD_N', lambda x: x.abs().sum()),
            gcop       =('IMPORTE_N',  lambda x: x.abs().sum() / 1e9),
            ordenes    =('ORDEN',      'nunique'),
        ).sort_values(w_metrica_y3.value, ascending=False).head(w_top_y3.value)
        print(f'Registros filtrados: {len(sub):,}  |  Grupos: {len(grp)}')
        display(grp.round(4))

for widget in [w_mat_y3, w_años_y3, w_agrupar_y3, w_metrica_y3, w_top_y3]:
    widget.observe(actualizar_y3, names='value')

panel_y3 = w.VBox([
    w.HBox([w_mat_y3, w_años_y3]),
    w.HBox([w_agrupar_y3, w_metrica_y3, w_top_y3]),
    out_y3
])
actualizar_y3()
panel_y3

## 3 · Detalle de una orden
Ingresa un número de orden para ver todos los materiales que consumió (receta real).

In [ ]:
w_orden = w.Text(
    value='', placeholder='ej: 000011265325',
    description='Orden:', layout=w.Layout(width='350px')
)
out_ord = w.Output()

def buscar_orden(_=None):
    with out_ord:
        clear_output(wait=True)
        orden = w_orden.value.strip().zfill(12)
        if not orden or orden == '000000000000':
            return
        sub = df[df['ORDEN'] == orden]
        if sub.empty:
            print(f'Orden {orden} no encontrada.')
            return
        print(f'Orden: {orden}  |  Fecha: {sub["FECHA"].min().date()} → {sub["FECHA"].max().date()}')
        receta = (
            sub.groupby(['MATERIAL', 'TEXTO_MATE', 'LOTE', 'UNIDAD'])
            .agg(
                m2       =('CANTIDAD_N', lambda x: x.abs().sum()),
                importe  =('IMPORTE_N',  lambda x: x.abs().sum()),
                lineas   =('ID',         'count'),
            )
            .reset_index()
            .sort_values('importe', ascending=False)
        )
        receta['es_y3'] = receta['TEXTO_MATE'].str.contains('y3', case=False, na=False)
        display(receta)
        total = receta['importe'].sum()
        print(f'Importe total: {total:,.0f} COP  ({total/1e6:.1f} MCOP)')

w_orden.observe(buscar_orden, names='value')
w.VBox([w_orden, out_ord])

## 4 · Análisis ad-hoc
`df` tiene `CANTIDAD_N`, `IMPORTE_N`, `FECHA`, `AÑO`, `MES` listos.

In [ ]:
# Ejemplo: consumo mensual de Y3 por tipo
(
    df[df['TEXTO_MATE'].str.contains('y3', case=False, na=False)]
    .groupby(['MES', 'TEXTO_MATE'])
    .agg(m2=('CANTIDAD_N', lambda x: x.abs().sum()))
    .reset_index()
    .pivot(index='MES', columns='TEXTO_MATE', values='m2')
    .fillna(0)
    .tail(24)  # últimos 24 meses
)